# Final Confirmatory Evaluation Runner

This notebook consumes one completed final confirmatory benchmark run and emits final aggregated reporting artifacts.

It enforces run_kind=final_confirmatory_benchmark and consumes fixed-loss outputs.

Run this from the standalone Final training folder after the training notebook has produced a results run.

In [1]:
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
)
from sklearn.preprocessing import label_binarize


def find_notebook_root(start: Path) -> Path:
    training_marker = Path("ml_model") / "notebooks" / "training" / "io.py"
    repo_markers = ("pyproject.toml", ".git")

    fallback = None
    for candidate in [start, *start.parents]:
        if not (candidate / training_marker).exists():
            continue
        if fallback is None:
            fallback = candidate
        if any((candidate / marker).exists() for marker in repo_markers):
            return candidate

    if fallback is not None:
        return fallback
    raise FileNotFoundError(f"Could not locate standalone final-training root from {start}")


NOTEBOOK_ROOT = find_notebook_root(Path.cwd().resolve())
if str(NOTEBOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_ROOT))

from ml_model.preprocessing.dataset_io import evaluation_dir, latest_run_dir, load_json, load_numpy_artifacts, save_csv, save_json
from ml_model.evaluation.metrics import (
    aggregate_numeric_columns,
    aggregate_per_class_metrics,
    evaluate_from_logits,
    save_confusion_matrix_artifacts,
    save_reliability_diagram_artifacts,
)

In [3]:
DATASET_VERSION = "v3_907k_cleaned"
ECE_N_BINS = 15
CONFIDENCE_THRESHOLDS = [0.5, 0.7, 0.8, 0.9]
EXPECTED_RUN_KIND = "final_confirmatory_benchmark"
FINAL_RESULTS_BASE_DIR_CANDIDATES = [
    NOTEBOOK_ROOT / "results",
    NOTEBOOK_ROOT / "ml_model" / "notebooks" / "training done" / "Final training" / "results",
    NOTEBOOK_ROOT / "ml_model" / "notebooks" / "training done" / "results",
]

run_dir_override = os.getenv("BENCHMARK_RUN_DIR", "").strip()
if run_dir_override:
    RUN_DIR = Path(run_dir_override).expanduser().resolve()
    FINAL_RESULTS_BASE_DIR = RUN_DIR.parent
else:
    final_dir_error = None
    RUN_DIR = None
    FINAL_RESULTS_BASE_DIR = FINAL_RESULTS_BASE_DIR_CANDIDATES[0]
    for candidate_base_dir in FINAL_RESULTS_BASE_DIR_CANDIDATES:
        try:
            RUN_DIR = latest_run_dir(
                base_dir=candidate_base_dir,
                dataset_version=DATASET_VERSION,
            )
            FINAL_RESULTS_BASE_DIR = candidate_base_dir
            break
        except FileNotFoundError as exc:
            final_dir_error = exc

    if RUN_DIR is None:
        searched = ", ".join(str(path) for path in FINAL_RESULTS_BASE_DIR_CANDIDATES)
        raise FileNotFoundError(
            f"No benchmark run directories found for dataset '{DATASET_VERSION}'. "
            f"Searched: {searched}"
        ) from final_dir_error

if not RUN_DIR.exists():
    raise FileNotFoundError(f"Run directory does not exist: {RUN_DIR}")

EVALUATION_OUTPUT_DIR = evaluation_dir(RUN_DIR)
manifest = load_json(RUN_DIR / "run_manifest.json")
if manifest.get("run_kind") != EXPECTED_RUN_KIND:
    raise ValueError(f"Expected run_kind={EXPECTED_RUN_KIND}, found {manifest.get('run_kind')}")

DATASET_VERSION = manifest.get("dataset_version", DATASET_VERSION)
FIXED_LOSS_KEY = str(manifest.get("fixed_loss_key", "")).strip()
model_keys = [str(x) for x in manifest.get("model_keys", [])] or sorted(
    p.name for p in RUN_DIR.iterdir() if p.is_dir() and p.name != "evaluation"
)

print(f"Run dir={RUN_DIR}")
print(f"Results base dir={FINAL_RESULTS_BASE_DIR}")
print(f"Evaluation dir={EVALUATION_OUTPUT_DIR}")
print(f"Models={model_keys}")
print(f"Fixed loss={FIXED_LOSS_KEY}")

Run dir=G:\AI\PDDDD\injection-alert-system\ml_model\notebooks\training done\Final training\results\v3_907k_cleaned_final_confirmatory_weighted_ce_3seed_20260412_035441
Results base dir=G:\AI\PDDDD\injection-alert-system\ml_model\notebooks\training done\Final training\results
Evaluation dir=G:\AI\PDDDD\injection-alert-system\ml_model\notebooks\training done\Final training\results\v3_907k_cleaned_final_confirmatory_weighted_ce_3seed_20260412_035441\evaluation
Models=['distilbert', 'minilm_l6', 'tinybert_bigru_attn']
Fixed loss=weighted_ce


In [4]:
HIGH_CONFIDENCE_TOP_N = 100


def parse_seed(seed_dir: Path) -> int:
    try:
        return int(seed_dir.name.split("_")[-1])
    except Exception:
        return -1


def coerce_float(value, default=np.nan) -> float:
    try:
        return float(value)
    except Exception:
        return float(default)


def resolve_loss_key(model_key: str) -> str:
    model_dir = RUN_DIR / model_key
    if FIXED_LOSS_KEY:
        expected = model_dir / f"loss_{FIXED_LOSS_KEY}"
        if expected.exists():
            return FIXED_LOSS_KEY
        raise FileNotFoundError(f"Missing expected fixed-loss dir: {expected}")

    loss_dirs = sorted([p for p in model_dir.glob("loss_*") if p.is_dir()])
    if len(loss_dirs) == 1:
        return loss_dirs[0].name.replace("loss_", "", 1)
    raise FileNotFoundError(f"Could not resolve fixed loss for model {model_key}")


def load_required_json(path: Path, description: str):
    if not path.exists():
        raise FileNotFoundError(f"Missing {description}: {path}")
    try:
        payload = load_json(path)
    except Exception as exc:
        raise RuntimeError(f"Failed to parse {description}: {path} ({exc})") from exc
    if payload is None:
        raise ValueError(f"Empty {description}: {path}")
    return payload


def load_required_outputs(path: Path, split_name: str) -> tuple[np.ndarray, np.ndarray]:
    if not path.exists():
        raise FileNotFoundError(f"Missing {split_name} outputs artifact: {path}")
    try:
        payload = load_numpy_artifacts(path)
    except Exception as exc:
        raise RuntimeError(f"Failed to load {split_name} outputs artifact: {path} ({exc})") from exc

    for key in ("logits", "labels"):
        if key not in payload:
            raise KeyError(f"Missing key '{key}' in {split_name} outputs artifact: {path}")

    logits = np.asarray(payload["logits"])
    labels = np.asarray(payload["labels"]).astype(np.int64)

    if logits.ndim != 2:
        raise ValueError(f"Expected 2D logits in {path}, got shape={logits.shape}")
    if labels.ndim != 1:
        raise ValueError(f"Expected 1D labels in {path}, got shape={labels.shape}")
    if logits.shape[0] != labels.shape[0]:
        raise ValueError(
            f"Mismatched rows between logits and labels in {path}: "
            f"{logits.shape[0]} vs {labels.shape[0]}"
        )
    if logits.shape[0] == 0:
        raise ValueError(f"No rows present in {split_name} outputs artifact: {path}")

    return logits, labels


def load_required_per_class_frame(path: Path, seed: int) -> pd.DataFrame:
    payload = load_required_json(path, "per-class metrics")
    frame = pd.DataFrame(payload)
    if frame.empty:
        raise ValueError(f"Per-class metrics artifact is empty: {path}")

    required_cols = {"label_id", "label_name", "precision", "recall", "f1", "support"}
    missing = sorted(required_cols - set(frame.columns))
    if missing:
        raise ValueError(f"Per-class metrics artifact missing columns {missing}: {path}")

    frame = frame.copy()
    frame["seed"] = int(seed)
    return frame


def update_label_map(label_map: dict[int, str], frame: pd.DataFrame) -> None:
    if frame.empty:
        return
    for _, row in frame[["label_id", "label_name"]].dropna().iterrows():
        label_id = int(row["label_id"])
        label_name = str(row["label_name"])
        if label_name:
            label_map[label_id] = label_name


def build_label_names(n_classes: int, label_map: dict[int, str], manifest_labels: list[str]) -> list[str]:
    names = []
    for idx in range(int(n_classes)):
        if idx in label_map:
            names.append(str(label_map[idx]))
        elif idx < len(manifest_labels) and str(manifest_labels[idx]).strip():
            names.append(str(manifest_labels[idx]).strip())
        else:
            names.append(f"class_{idx}")
    return names


def find_normal_label_id(label_names: list[str]) -> int | None:
    for idx, name in enumerate(label_names):
        if str(name).strip().lower() == "normal":
            return int(idx)
    return None


def one_hot_labels(labels: np.ndarray, n_classes: int) -> np.ndarray:
    encoded = label_binarize(labels, classes=np.arange(int(n_classes)))
    encoded = np.asarray(encoded)

    if encoded.ndim == 1:
        encoded = encoded.reshape(-1, 1)

    if encoded.shape[1] == 1 and int(n_classes) == 2:
        encoded = np.hstack([1 - encoded, encoded])

    if encoded.shape[1] != int(n_classes):
        repaired = np.zeros((labels.shape[0], int(n_classes)), dtype=np.int64)
        for idx, label in enumerate(labels.astype(np.int64)):
            if 0 <= int(label) < int(n_classes):
                repaired[idx, int(label)] = 1
        encoded = repaired

    return encoded


def compute_macro_roc_auc_ovr(labels: np.ndarray, probs: np.ndarray) -> float:
    n_classes = int(probs.shape[1])
    try:
        return float(
            roc_auc_score(
                labels,
                probs,
                labels=np.arange(n_classes),
                multi_class="ovr",
                average="macro",
            )
        )
    except Exception:
        return float(np.nan)


def compute_macro_average_precision_ovr(labels: np.ndarray, probs: np.ndarray) -> float:
    n_classes = int(probs.shape[1])
    try:
        y_true = one_hot_labels(labels, n_classes)
        return float(average_precision_score(y_true, probs, average="macro"))
    except Exception:
        return float(np.nan)


def compute_per_class_roc_pr_rows(labels: np.ndarray, probs: np.ndarray, label_names: list[str]) -> list[dict[str, float | int | str]]:
    n_classes = int(probs.shape[1])
    y_true = one_hot_labels(labels, n_classes)
    rows: list[dict[str, float | int | str]] = []

    for label_id in range(n_classes):
        binary_true = y_true[:, label_id]
        scores = probs[:, label_id]
        positives = int(np.sum(binary_true))

        if positives == 0 or positives == int(binary_true.shape[0]):
            roc_auc = float(np.nan)
            avg_precision = float(np.nan)
        else:
            try:
                roc_auc = float(roc_auc_score(binary_true, scores))
            except Exception:
                roc_auc = float(np.nan)
            try:
                avg_precision = float(average_precision_score(binary_true, scores))
            except Exception:
                avg_precision = float(np.nan)

        rows.append(
            {
                "label_id": int(label_id),
                "label_name": str(label_names[label_id]) if label_id < len(label_names) else f"class_{label_id}",
                "support": positives,
                "roc_auc_ovr": roc_auc,
                "average_precision_ovr": avg_precision,
            }
        )

    return rows


def compute_threshold_rows(
    labels: np.ndarray,
    preds: np.ndarray,
    probs: np.ndarray,
    thresholds: list[float],
    normal_label_id: int | None,
) -> list[dict[str, float | int]]:
    confidences = probs.max(axis=1)
    total = max(int(labels.shape[0]), 1)
    rows: list[dict[str, float | int]] = []

    for threshold in thresholds:
        threshold_value = float(threshold)
        accepted = confidences >= threshold_value
        kept = int(np.sum(accepted))
        coverage = float(kept / total)

        if kept == 0:
            row = {
                "threshold": threshold_value,
                "count_retained": 0,
                "coverage": coverage,
                "accuracy": np.nan,
                "balanced_accuracy": np.nan,
                "macro_f1": np.nan,
                "normal_false_positive_rate": np.nan,
                "attack_escape_rate": np.nan,
            }
            rows.append(row)
            continue

        y_true = labels[accepted]
        y_pred = preds[accepted]
        acc = float(accuracy_score(y_true, y_pred))
        balanced_acc = float(balanced_accuracy_score(y_true, y_pred))
        macro_f1 = float(f1_score(y_true, y_pred, average="macro", zero_division=0))

        if normal_label_id is None:
            normal_false_positive_rate = float(np.nan)
            attack_escape_rate = float(np.nan)
        else:
            normal_mask = y_true == int(normal_label_id)
            attack_mask = y_true != int(normal_label_id)

            normal_total = int(np.sum(normal_mask))
            attack_total = int(np.sum(attack_mask))

            normal_predicted_attack = int(np.sum(normal_mask & (y_pred != int(normal_label_id))))
            attack_predicted_normal = int(np.sum(attack_mask & (y_pred == int(normal_label_id))))

            normal_false_positive_rate = (
                float(normal_predicted_attack / normal_total) if normal_total > 0 else float(np.nan)
            )
            attack_escape_rate = float(attack_predicted_normal / attack_total) if attack_total > 0 else float(np.nan)

        rows.append(
            {
                "threshold": threshold_value,
                "count_retained": kept,
                "coverage": coverage,
                "accuracy": acc,
                "balanced_accuracy": balanced_acc,
                "macro_f1": macro_f1,
                "normal_false_positive_rate": normal_false_positive_rate,
                "attack_escape_rate": attack_escape_rate,
            }
        )

    return rows


def compute_high_confidence_error_rows(
    labels: np.ndarray,
    preds: np.ndarray,
    probs: np.ndarray,
    seed: int,
    label_names: list[str],
) -> list[dict[str, float | int | str]]:
    confidences = probs.max(axis=1)
    incorrect_indices = np.where(preds != labels)[0]
    rows: list[dict[str, float | int | str]] = []

    for sample_index in incorrect_indices:
        true_id = int(labels[sample_index])
        pred_id = int(preds[sample_index])
        rows.append(
            {
                "seed": int(seed),
                "sample_index": int(sample_index),
                "true_label_id": true_id,
                "true_label": label_names[true_id] if true_id < len(label_names) else f"class_{true_id}",
                "predicted_label_id": pred_id,
                "predicted_label": label_names[pred_id] if pred_id < len(label_names) else f"class_{pred_id}",
                "confidence": float(confidences[sample_index]),
            }
        )

    return rows


def find_calibration_worsening(test_uncal: dict, test_cal: dict) -> list[str]:
    worsening = []
    if coerce_float(test_cal.get("ece")) > coerce_float(test_uncal.get("ece")):
        worsening.append("ece")
    if coerce_float(test_cal.get("nll")) > coerce_float(test_uncal.get("nll")):
        worsening.append("nll")
    if coerce_float(test_cal.get("brier_score")) > coerce_float(test_uncal.get("brier_score")):
        worsening.append("brier_score")
    return worsening


def save_output_csv(df: pd.DataFrame, path: Path, generated_outputs: set[str], *, index: bool = False) -> None:
    save_csv(df, path, index=index)
    generated_outputs.add(Path(path).name)


def save_output_json(path: Path, payload: dict, generated_outputs: set[str]) -> None:
    save_json(path, payload)
    generated_outputs.add(Path(path).name)


def require_summary_str(summary: dict, field: str, model_key: str, seed: int) -> str:
    value = str(summary.get(field, "")).strip()
    if not value:
        raise ValueError(f"Missing or empty summary field '{field}' for {model_key} seed={seed}")
    return value


def require_summary_float(summary: dict, field: str, model_key: str, seed: int) -> float:
    value = coerce_float(summary.get(field), default=np.nan)
    if not np.isfinite(value):
        raise ValueError(f"Missing or invalid numeric summary field '{field}' for {model_key} seed={seed}")
    return value


def mean_normalized_confusion_matrix_frame(
    labels_by_seed: list[np.ndarray],
    preds_by_seed: list[np.ndarray],
    label_names: list[str],
) -> pd.DataFrame:
    if not labels_by_seed or not preds_by_seed:
        raise ValueError("Cannot build aggregate confusion matrix without per-seed predictions")
    if len(labels_by_seed) != len(preds_by_seed):
        raise ValueError("Mismatched labels/predictions seed lists for confusion aggregation")

    label_ids = np.arange(len(label_names), dtype=np.int64)
    matrices = []
    for labels, preds in zip(labels_by_seed, preds_by_seed):
        cm = confusion_matrix(labels, preds, labels=label_ids, normalize="true")
        matrices.append(np.asarray(cm, dtype=np.float64))

    mean_cm = np.mean(np.stack(matrices, axis=0), axis=0)
    return pd.DataFrame(mean_cm, index=label_names, columns=label_names)


def save_mean_normalized_confusion_matrix_artifacts(
    cm_df: pd.DataFrame,
    csv_path: Path,
    png_path: Path | None,
    title: str,
    generated_outputs: set[str],
) -> None:
    csv_path = Path(csv_path)
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    cm_df.to_csv(csv_path)
    generated_outputs.add(csv_path.name)

    if png_path is None:
        return

    png_path = Path(png_path)
    png_path.parent.mkdir(parents=True, exist_ok=True)
    try:
        import matplotlib.pyplot as plt

        fig, ax = plt.subplots(figsize=(8, 6))
        cm_values = np.asarray(cm_df.values, dtype=np.float64)
        im = ax.imshow(cm_values, cmap="Blues", vmin=0.0, vmax=1.0)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        ax.set_xticks(range(cm_df.shape[1]), cm_df.columns, rotation=45, ha="right")
        ax.set_yticks(range(cm_df.shape[0]), cm_df.index)
        for i in range(cm_df.shape[0]):
            for j in range(cm_df.shape[1]):
                ax.text(j, i, f"{cm_values[i, j]:.3f}", ha="center", va="center", color="black")
        ax.set_xlabel("Predicted label")
        ax.set_ylabel("True label")
        ax.set_title(title)
        fig.tight_layout()
        fig.savefig(str(png_path), dpi=200)
        plt.close(fig)
        generated_outputs.add(png_path.name)
    except Exception as exc:
        error_path = png_path.with_name(f"{png_path.stem}_png_error.txt")
        error_path.write_text(f"error_type: {type(exc).__name__}\nerror_message: {exc}\n", encoding="utf-8")
        generated_outputs.add(error_path.name)


comparison_rows = []
per_class_tables = []
latency_rows = []
threshold_rows_all = []
roc_pr_rows_all = []
consumed_artifacts = set()
generated_outputs = set()
latency_warnings = []
calibration_warnings = []
artifact_warnings = []

manifest_label_names = [str(x) for x in manifest.get("label_names", [])]
model_seed_counts: dict[str, int] = {}
model_eval_cache: dict[tuple[str, str], dict[str, np.ndarray | list[str]]] = {}

for model_key in model_keys:
    loss_key = resolve_loss_key(model_key)

    variant_dir = RUN_DIR / model_key / f"loss_{loss_key}"
    seed_dirs = sorted([p for p in variant_dir.glob("seed_*") if p.is_dir()])
    if not seed_dirs:
        raise FileNotFoundError(f"No seed dirs under {variant_dir}")

    model_seed_counts[model_key] = int(len(seed_dirs))

    seed_rows = []
    per_class_seed_frames = []
    latency_seed_rows = []
    model_threshold_rows = []
    model_roc_pr_rows = []
    model_high_conf_errors = []

    model_label_map: dict[int, str] = {}
    model_test_labels_all = []
    model_test_preds_all = []
    model_test_probs_uncal_all = []
    model_test_probs_cal_all = []

    for seed_dir in seed_dirs:
        seed = parse_seed(seed_dir)

        summary_path = seed_dir / "summary_metrics.json"
        calibration_path = seed_dir / "calibration.json"
        validation_outputs_path = seed_dir / "validation_outputs.npz"
        test_outputs_path = seed_dir / "test_outputs.npz"
        per_class_path = seed_dir / "per_class_metrics.json"
        latency_path = seed_dir / "latency_summary.json"

        summary = load_required_json(summary_path, "summary metrics")
        consumed_artifacts.add(str(summary_path))
        if not isinstance(summary, dict):
            raise ValueError(f"Summary metrics payload must be a JSON object: {summary_path}")

        required_summary_fields = [
            "architecture",
            "architecture_family",
            "head_type",
            "experiment_phase",
            "normal_false_positive_rate",
            "attack_escape_rate",
            "inference_latency_mean_ms",
            "inference_latency_std_ms",
            "inference_latency_p50_ms",
            "inference_latency_p95_ms",
        ]
        missing_summary_fields = [field for field in required_summary_fields if field not in summary]
        if missing_summary_fields:
            raise KeyError(
                f"Missing required summary fields for {model_key} seed={seed}: {missing_summary_fields}"
            )

        architecture = require_summary_str(summary, "architecture", model_key, seed)
        architecture_family = require_summary_str(summary, "architecture_family", model_key, seed)
        head_type = require_summary_str(summary, "head_type", model_key, seed)
        experiment_phase = require_summary_str(summary, "experiment_phase", model_key, seed)
        summary_normal_false_positive_rate = require_summary_float(summary, "normal_false_positive_rate", model_key, seed)
        summary_attack_escape_rate = require_summary_float(summary, "attack_escape_rate", model_key, seed)
        summary_inference_latency_mean_ms = require_summary_float(summary, "inference_latency_mean_ms", model_key, seed)
        summary_inference_latency_std_ms = require_summary_float(summary, "inference_latency_std_ms", model_key, seed)
        summary_inference_latency_p50_ms = require_summary_float(summary, "inference_latency_p50_ms", model_key, seed)
        summary_inference_latency_p95_ms = require_summary_float(summary, "inference_latency_p95_ms", model_key, seed)

        calibration = load_required_json(calibration_path, "calibration")
        consumed_artifacts.add(str(calibration_path))
        if not isinstance(calibration, dict):
            raise ValueError(f"Calibration payload must be a JSON object: {calibration_path}")

        temperature = coerce_float(calibration.get("temperature"), default=np.nan)
        if not np.isfinite(temperature) or temperature <= 0:
            raise ValueError(f"Invalid calibration temperature for {model_key} seed={seed}: {temperature}")

        val_logits, val_labels = load_required_outputs(validation_outputs_path, "validation")
        test_logits, test_labels = load_required_outputs(test_outputs_path, "test")
        consumed_artifacts.add(str(validation_outputs_path))
        consumed_artifacts.add(str(test_outputs_path))

        per_class_seed_frame = load_required_per_class_frame(per_class_path, seed=seed)
        consumed_artifacts.add(str(per_class_path))
        per_class_seed_frames.append(per_class_seed_frame)
        update_label_map(model_label_map, per_class_seed_frame)

        n_classes = int(test_logits.shape[1])
        label_names = build_label_names(n_classes, model_label_map, manifest_label_names)
        normal_label_id = find_normal_label_id(label_names)

        val_uncal = evaluate_from_logits(val_logits, val_labels, n_bins=ECE_N_BINS)
        val_cal = evaluate_from_logits(val_logits / temperature, val_labels, n_bins=ECE_N_BINS)
        test_uncal = evaluate_from_logits(test_logits, test_labels, n_bins=ECE_N_BINS)
        test_cal = evaluate_from_logits(test_logits / temperature, test_labels, n_bins=ECE_N_BINS)

        test_mcc = coerce_float(matthews_corrcoef(test_labels, test_uncal["preds"]))
        test_macro_roc_auc_ovr = compute_macro_roc_auc_ovr(test_labels, test_uncal["probs"])
        test_macro_average_precision_ovr = compute_macro_average_precision_ovr(test_labels, test_uncal["probs"])

        calibration_worsening = find_calibration_worsening(test_uncal, test_cal)
        if calibration_worsening:
            warning = (
                f"Calibration worsened for {model_key} seed={seed} on "
                f"{', '.join(calibration_worsening)}"
            )
            print(f"WARNING: {warning}")
            calibration_warnings.append(
                {
                    "model_key": model_key,
                    "loss_key": loss_key,
                    "seed": int(seed),
                    "worsened_metrics": calibration_worsening,
                }
            )

        seed_rows.append(
            {
                "model_key": model_key,
                "loss_key": loss_key,
                "seed": int(seed),
                "architecture": architecture,
                "architecture_family": architecture_family,
                "head_type": head_type,
                "experiment_phase": experiment_phase,
                "temperature": coerce_float(temperature),
                "val_accuracy": coerce_float(val_uncal["accuracy"]),
                "val_balanced_accuracy": coerce_float(val_uncal["balanced_accuracy"]),
                "val_macro_f1": coerce_float(val_uncal["macro_f1"]),
                "val_weighted_f1": coerce_float(val_uncal["weighted_f1"]),
                "val_ece_uncalibrated": coerce_float(val_uncal["ece"]),
                "val_ece_calibrated": coerce_float(val_cal["ece"]),
                "val_nll_uncalibrated": coerce_float(val_uncal["nll"]),
                "val_nll_calibrated": coerce_float(val_cal["nll"]),
                "val_brier_uncalibrated": coerce_float(val_uncal["brier_score"]),
                "val_brier_calibrated": coerce_float(val_cal["brier_score"]),
                "test_accuracy": coerce_float(test_uncal["accuracy"]),
                "test_balanced_accuracy": coerce_float(test_uncal["balanced_accuracy"]),
                "test_macro_f1": coerce_float(test_uncal["macro_f1"]),
                "test_weighted_f1": coerce_float(test_uncal["weighted_f1"]),
                "test_ece_uncalibrated": coerce_float(test_uncal["ece"]),
                "test_ece_calibrated": coerce_float(test_cal["ece"]),
                "test_nll_uncalibrated": coerce_float(test_uncal["nll"]),
                "test_nll_calibrated": coerce_float(test_cal["nll"]),
                "test_brier_uncalibrated": coerce_float(test_uncal["brier_score"]),
                "test_brier_calibrated": coerce_float(test_cal["brier_score"]),
                "test_mcc": test_mcc,
                "test_macro_roc_auc_ovr": test_macro_roc_auc_ovr,
                "test_macro_average_precision_ovr": test_macro_average_precision_ovr,
                "normal_false_positive_rate": summary_normal_false_positive_rate,
                "attack_escape_rate": summary_attack_escape_rate,
                "inference_latency_mean_ms": summary_inference_latency_mean_ms,
                "inference_latency_std_ms": summary_inference_latency_std_ms,
                "inference_latency_p50_ms": summary_inference_latency_p50_ms,
                "inference_latency_p95_ms": summary_inference_latency_p95_ms,
                "training_workflow_runtime_sec": coerce_float(summary.get("training_workflow_runtime_sec")),
                "mean_epoch_training_time_sec": coerce_float(summary.get("mean_epoch_training_time_sec")),
            }
        )

        threshold_rows = compute_threshold_rows(
            labels=test_labels,
            preds=test_uncal["preds"],
            probs=test_uncal["probs"],
            thresholds=[float(t) for t in CONFIDENCE_THRESHOLDS],
            normal_label_id=normal_label_id,
        )
        for row in threshold_rows:
            row["model_key"] = model_key
            row["loss_key"] = loss_key
            row["seed"] = int(seed)
        model_threshold_rows.extend(threshold_rows)
        threshold_rows_all.extend(threshold_rows)

        per_class_roc_pr_rows = compute_per_class_roc_pr_rows(test_labels, test_uncal["probs"], label_names)
        for row in per_class_roc_pr_rows:
            row["model_key"] = model_key
            row["loss_key"] = loss_key
            row["seed"] = int(seed)
        model_roc_pr_rows.extend(per_class_roc_pr_rows)
        roc_pr_rows_all.extend(per_class_roc_pr_rows)

        model_high_conf_errors.extend(
            compute_high_confidence_error_rows(
                labels=test_labels,
                preds=test_uncal["preds"],
                probs=test_uncal["probs"],
                seed=int(seed),
                label_names=label_names,
            )
        )

        if latency_path.exists():
            latency_payload = load_required_json(latency_path, "latency summary")
            consumed_artifacts.add(str(latency_path))
            if not isinstance(latency_payload, dict):
                raise ValueError(f"Latency summary payload must be a JSON object: {latency_path}")

            latency_row = {
                "seed": int(seed),
                "inference_latency_mean_ms": coerce_float(latency_payload.get("latency_mean_ms")),
                "inference_latency_std_ms": coerce_float(latency_payload.get("latency_std_ms")),
                "inference_latency_p50_ms": coerce_float(latency_payload.get("latency_p50_ms")),
                "inference_latency_p95_ms": coerce_float(latency_payload.get("latency_p95_ms")),
                "inference_latency_min_ms": coerce_float(latency_payload.get("latency_min_ms")),
                "inference_latency_max_ms": coerce_float(latency_payload.get("latency_max_ms")),
                "latency_n_measurements": coerce_float(latency_payload.get("n_measurements")),
            }
            latency_seed_rows.append(latency_row)

            null_fields = [
                key
                for key, value in latency_row.items()
                if key != "seed" and not np.isfinite(coerce_float(value, default=np.nan))
            ]
            if null_fields:
                warning = (
                    f"Latency artifact has null/NaN values for {model_key} seed={seed}: "
                    f"{', '.join(null_fields)}"
                )
                print(f"WARNING: {warning}")
                latency_warnings.append(warning)
        else:
            warning = f"Missing latency summary artifact for {model_key} seed={seed}: {latency_path}"
            print(f"WARNING: {warning}")
            latency_warnings.append(warning)

        model_test_labels_all.append(test_labels)
        model_test_preds_all.append(test_uncal["preds"])
        model_test_probs_uncal_all.append(np.asarray(test_uncal["probs"], dtype=np.float64))
        model_test_probs_cal_all.append(np.asarray(test_cal["probs"], dtype=np.float64))

    seed_df = pd.DataFrame(seed_rows).sort_values(by="seed").reset_index(drop=True)
    if seed_df.empty:
        raise RuntimeError(f"No seed metrics generated for {model_key}")

    save_output_csv(seed_df, EVALUATION_OUTPUT_DIR / f"{model_key}_{loss_key}_seed_metrics.csv", generated_outputs, index=False)

    comparison_rows.append(
        {
            "model_key": model_key,
            "loss_key": loss_key,
            "n_seeds": int(seed_df.shape[0]),
            "architecture": str(seed_df.iloc[0]["architecture"]),
            "architecture_family": str(seed_df.iloc[0]["architecture_family"]),
            "head_type": str(seed_df.iloc[0]["head_type"]),
            "experiment_phase": str(seed_df.iloc[0]["experiment_phase"]),
            **aggregate_numeric_columns(seed_df, exclude=["seed"]),
        }
    )

    per_class_df = aggregate_per_class_metrics(per_class_seed_frames)
    if per_class_df.empty:
        raise RuntimeError(f"Per-class aggregation returned empty frame for {model_key}")
    per_class_df.insert(0, "loss_key", loss_key)
    per_class_df.insert(0, "model_key", model_key)
    per_class_tables.append(per_class_df)

    if latency_seed_rows:
        latency_seed_df = pd.DataFrame(latency_seed_rows).sort_values(by="seed").reset_index(drop=True)
        save_output_csv(
            latency_seed_df,
            EVALUATION_OUTPUT_DIR / f"{model_key}_{loss_key}_latency_seed_metrics.csv",
            generated_outputs,
            index=False,
        )
        latency_rows.append(
            {
                "model_key": model_key,
                "loss_key": loss_key,
                "n_seeds": int(latency_seed_df.shape[0]),
                **aggregate_numeric_columns(latency_seed_df, exclude=["seed"]),
            }
        )
    else:
        warning = f"No usable latency rows available for {model_key}"
        print(f"WARNING: {warning}")
        latency_warnings.append(warning)

    model_threshold_df = pd.DataFrame(model_threshold_rows).sort_values(by=["seed", "threshold"]).reset_index(drop=True)
    save_output_csv(
        model_threshold_df,
        EVALUATION_OUTPUT_DIR / f"{model_key}_{loss_key}_threshold_analysis.csv",
        generated_outputs,
        index=False,
    )

    model_roc_pr_df = pd.DataFrame(model_roc_pr_rows).sort_values(by=["seed", "label_id"]).reset_index(drop=True)
    save_output_csv(
        model_roc_pr_df,
        EVALUATION_OUTPUT_DIR / f"{model_key}_{loss_key}_per_class_roc_pr_seed_metrics.csv",
        generated_outputs,
        index=False,
    )

    model_roc_pr_summary = (
        model_roc_pr_df.groupby(["model_key", "loss_key", "label_id", "label_name"], as_index=False)
        .agg(
            support_mean=("support", "mean"),
            support_std=("support", "std"),
            roc_auc_ovr_mean=("roc_auc_ovr", "mean"),
            roc_auc_ovr_std=("roc_auc_ovr", "std"),
            average_precision_ovr_mean=("average_precision_ovr", "mean"),
            average_precision_ovr_std=("average_precision_ovr", "std"),
        )
        .sort_values(by=["model_key", "label_id"]) 
        .reset_index(drop=True)
    )
    save_output_csv(
        model_roc_pr_summary,
        EVALUATION_OUTPUT_DIR / f"{model_key}_{loss_key}_per_class_roc_pr_summary.csv",
        generated_outputs,
        index=False,
    )

    if model_high_conf_errors:
        model_error_df = (
            pd.DataFrame(model_high_conf_errors)
            .sort_values(by="confidence", ascending=False)
            .head(HIGH_CONFIDENCE_TOP_N)
            .reset_index(drop=True)
        )
    else:
        model_error_df = pd.DataFrame(
            columns=[
                "seed",
                "sample_index",
                "true_label_id",
                "true_label",
                "predicted_label_id",
                "predicted_label",
                "confidence",
            ]
        )
    save_output_csv(
        model_error_df,
        EVALUATION_OUTPUT_DIR / f"{model_key}_high_confidence_errors.csv",
        generated_outputs,
        index=False,
    )

    aggregate_test_labels = np.concatenate(model_test_labels_all, axis=0)
    aggregate_test_preds = np.concatenate(model_test_preds_all, axis=0)
    aggregate_test_probs_uncal = np.concatenate(model_test_probs_uncal_all, axis=0)
    aggregate_test_probs_cal = np.concatenate(model_test_probs_cal_all, axis=0)

    aggregate_label_names = build_label_names(
        int(aggregate_test_probs_uncal.shape[1]),
        model_label_map,
        manifest_label_names,
    )

    mean_normalized_cm_df = mean_normalized_confusion_matrix_frame(
        labels_by_seed=model_test_labels_all,
        preds_by_seed=model_test_preds_all,
        label_names=aggregate_label_names,
    )
    save_mean_normalized_confusion_matrix_artifacts(
        cm_df=mean_normalized_cm_df,
        csv_path=EVALUATION_OUTPUT_DIR / f"{model_key}_aggregate_confusion_matrix.csv",
        png_path=EVALUATION_OUTPUT_DIR / f"{model_key}_aggregate_confusion_matrix.png",
        title=f"{model_key} mean normalized confusion matrix across seeds",
        generated_outputs=generated_outputs,
    )

    save_reliability_diagram_artifacts(
        probs=aggregate_test_probs_uncal,
        labels=aggregate_test_labels,
        csv_path=EVALUATION_OUTPUT_DIR / f"{model_key}_reliability_uncalibrated.csv",
        png_path=EVALUATION_OUTPUT_DIR / f"{model_key}_reliability_uncalibrated.png",
        n_bins=ECE_N_BINS,
        title=f"{model_key} reliability (uncalibrated)",
    )
    generated_outputs.add(f"{model_key}_reliability_uncalibrated.csv")
    generated_outputs.add(f"{model_key}_reliability_uncalibrated.png")

    save_reliability_diagram_artifacts(
        probs=aggregate_test_probs_cal,
        labels=aggregate_test_labels,
        csv_path=EVALUATION_OUTPUT_DIR / f"{model_key}_reliability_calibrated.csv",
        png_path=EVALUATION_OUTPUT_DIR / f"{model_key}_reliability_calibrated.png",
        n_bins=ECE_N_BINS,
        title=f"{model_key} reliability (calibrated)",
    )
    generated_outputs.add(f"{model_key}_reliability_calibrated.csv")
    generated_outputs.add(f"{model_key}_reliability_calibrated.png")

    model_eval_cache[(model_key, loss_key)] = {
        "labels": aggregate_test_labels,
        "probs_uncal": aggregate_test_probs_uncal,
        "probs_cal": aggregate_test_probs_cal,
        "label_names": aggregate_label_names,
    }

if not comparison_rows:
    raise RuntimeError("No completed final confirmatory artifacts were available for evaluation.")

seed_count_set = sorted(set(model_seed_counts.values()))
if len(seed_count_set) != 1:
    details = ", ".join(f"{key}:{value}" for key, value in sorted(model_seed_counts.items()))
    raise RuntimeError(f"Seed-count parity check failed across models ({details})")
print(f"Seed-count parity check passed: {seed_count_set[0]} seeds per model")

final_model_comparison = (
    pd.DataFrame(comparison_rows)
    .sort_values(by=["test_macro_f1_mean", "test_balanced_accuracy_mean", "test_accuracy_mean"], ascending=[False, False, False])
    .reset_index(drop=True)
)
save_output_csv(final_model_comparison, EVALUATION_OUTPUT_DIR / "final_model_comparison.csv", generated_outputs, index=False)

aggregated_per_class = (
    pd.concat(per_class_tables, ignore_index=True).sort_values(by=["model_key", "label_id"]).reset_index(drop=True)
    if per_class_tables
    else pd.DataFrame()
)
save_output_csv(aggregated_per_class, EVALUATION_OUTPUT_DIR / "aggregated_per_class_summary.csv", generated_outputs, index=False)

latency_comparison = (
    pd.DataFrame(latency_rows)
    .sort_values(by=["inference_latency_mean_ms_mean", "model_key"], ascending=[True, True])
    .reset_index(drop=True)
    if latency_rows
    else pd.DataFrame()
)
save_output_csv(latency_comparison, EVALUATION_OUTPUT_DIR / "latency_comparison.csv", generated_outputs, index=False)

threshold_analysis_df = (
    pd.DataFrame(threshold_rows_all)
    .sort_values(by=["model_key", "seed", "threshold"]) 
    .reset_index(drop=True)
)
save_output_csv(threshold_analysis_df, EVALUATION_OUTPUT_DIR / "threshold_analysis.csv", generated_outputs, index=False)

threshold_summary_df = (
    threshold_analysis_df.groupby(["model_key", "loss_key", "threshold"], as_index=False)
    .agg(
        coverage_mean=("coverage", "mean"),
        coverage_std=("coverage", "std"),
        accuracy_mean=("accuracy", "mean"),
        accuracy_std=("accuracy", "std"),
        balanced_accuracy_mean=("balanced_accuracy", "mean"),
        balanced_accuracy_std=("balanced_accuracy", "std"),
        macro_f1_mean=("macro_f1", "mean"),
        macro_f1_std=("macro_f1", "std"),
        normal_false_positive_rate_mean=("normal_false_positive_rate", "mean"),
        normal_false_positive_rate_std=("normal_false_positive_rate", "std"),
        attack_escape_rate_mean=("attack_escape_rate", "mean"),
        attack_escape_rate_std=("attack_escape_rate", "std"),
    )
    .sort_values(by=["model_key", "threshold"]) 
    .reset_index(drop=True)
)
save_output_csv(threshold_summary_df, EVALUATION_OUTPUT_DIR / "threshold_analysis_summary.csv", generated_outputs, index=False)

roc_pr_seed_df = (
    pd.DataFrame(roc_pr_rows_all)
    .sort_values(by=["model_key", "seed", "label_id"]) 
    .reset_index(drop=True)
)
save_output_csv(roc_pr_seed_df, EVALUATION_OUTPUT_DIR / "roc_pr_per_class_seed_metrics.csv", generated_outputs, index=False)

roc_pr_summary_df = (
    roc_pr_seed_df.groupby(["model_key", "loss_key", "label_id", "label_name"], as_index=False)
    .agg(
        support_mean=("support", "mean"),
        support_std=("support", "std"),
        roc_auc_ovr_mean=("roc_auc_ovr", "mean"),
        roc_auc_ovr_std=("roc_auc_ovr", "std"),
        average_precision_ovr_mean=("average_precision_ovr", "mean"),
        average_precision_ovr_std=("average_precision_ovr", "std"),
    )
    .sort_values(by=["model_key", "label_id"]) 
    .reset_index(drop=True)
)
save_output_csv(roc_pr_summary_df, EVALUATION_OUTPUT_DIR / "roc_pr_per_class_summary.csv", generated_outputs, index=False)

top_row = final_model_comparison.iloc[0]
runner_up_row = final_model_comparison.iloc[1] if final_model_comparison.shape[0] > 1 else None

primary_metric = "test_macro_f1_mean"
top_model_key = str(top_row["model_key"])
top_loss_key = str(top_row["loss_key"])
top_score = coerce_float(top_row.get(primary_metric))

if runner_up_row is not None:
    runner_up_score = coerce_float(runner_up_row.get(primary_metric))
    score_gap = float(top_score - runner_up_score)
else:
    runner_up_score = float(np.nan)
    score_gap = float(np.nan)

results_are_close = bool(np.isfinite(score_gap) and score_gap <= 0.005)
tradeoff_note = "Top model has the highest macro F1 with current evidence."
if results_are_close:
    tradeoff_note = "Top models are close across seeds; prioritize calibration, latency, and security tradeoffs."

latency_warning_for_top = [w for w in latency_warnings if top_model_key in w]
calibration_warning_for_top = [w for w in calibration_warnings if w.get("model_key") == top_model_key]

recommendation_payload = {
    "primary_metric": primary_metric,
    "recommended_model": {
        "model_key": top_model_key,
        "loss_key": top_loss_key,
        "test_macro_f1_mean": top_score,
        "test_macro_f1_std": coerce_float(top_row.get("test_macro_f1_std")),
        "test_balanced_accuracy_mean": coerce_float(top_row.get("test_balanced_accuracy_mean")),
        "test_accuracy_mean": coerce_float(top_row.get("test_accuracy_mean")),
    },
    "runner_up": (
        {
            "model_key": str(runner_up_row.get("model_key")),
            "loss_key": str(runner_up_row.get("loss_key")),
            "test_macro_f1_mean": runner_up_score,
        }
        if runner_up_row is not None
        else None
    ),
    "macro_f1_gap_to_runner_up": score_gap,
    "results_are_close": results_are_close,
    "tradeoff_note": tradeoff_note,
    "latency_warnings_for_recommended_model": latency_warning_for_top,
    "calibration_warnings_for_recommended_model": calibration_warning_for_top,
    "all_latency_warnings": latency_warnings,
    "all_calibration_warnings": calibration_warnings,
    "created_at": datetime.now(timezone.utc).isoformat(),
}
save_output_json(EVALUATION_OUTPUT_DIR / "model_recommendation.json", recommendation_payload, generated_outputs)

if (top_model_key, top_loss_key) in model_eval_cache:
    top_cache = model_eval_cache[(top_model_key, top_loss_key)]
    save_reliability_diagram_artifacts(
        probs=np.asarray(top_cache["probs_uncal"]),
        labels=np.asarray(top_cache["labels"]),
        csv_path=EVALUATION_OUTPUT_DIR / "reliability_uncalibrated.csv",
        png_path=EVALUATION_OUTPUT_DIR / "reliability_uncalibrated.png",
        n_bins=ECE_N_BINS,
        title=f"{top_model_key} reliability (uncalibrated)",
    )
    generated_outputs.add("reliability_uncalibrated.csv")
    generated_outputs.add("reliability_uncalibrated.png")

    save_reliability_diagram_artifacts(
        probs=np.asarray(top_cache["probs_cal"]),
        labels=np.asarray(top_cache["labels"]),
        csv_path=EVALUATION_OUTPUT_DIR / "reliability_calibrated.csv",
        png_path=EVALUATION_OUTPUT_DIR / "reliability_calibrated.png",
        n_bins=ECE_N_BINS,
        title=f"{top_model_key} reliability (calibrated)",
    )
    generated_outputs.add("reliability_calibrated.csv")
    generated_outputs.add("reliability_calibrated.png")

summary_payload = {
    "evaluation_kind": "final_confirmatory_evaluation",
    "source_run_kind": manifest.get("run_kind"),
    "source_run_dir": str(RUN_DIR),
    "evaluation_output_dir": str(EVALUATION_OUTPUT_DIR),
    "dataset_version": DATASET_VERSION,
    "metric_schema_version": manifest.get("artifact_schema_version", "final_confirmatory_benchmark.v1"),
    "model_count": int(final_model_comparison.shape[0]),
    "seed_counts": model_seed_counts,
    "confidence_thresholds": [float(t) for t in CONFIDENCE_THRESHOLDS],
    "aggregate_confusion_matrix_definition": "mean row-normalized confusion matrix averaged across seeds on repeated test split",
    "generated_outputs": sorted(generated_outputs),
    "consumed_artifacts": sorted(consumed_artifacts),
    "warnings": {
        "artifact_warnings": artifact_warnings,
        "latency_warnings": latency_warnings,
        "calibration_warnings": calibration_warnings,
    },
    "created_at": datetime.now(timezone.utc).isoformat(),
}
save_output_json(EVALUATION_OUTPUT_DIR / "evaluation_summary.json", summary_payload, generated_outputs)

comparison_display_columns = [
    "model_key",
    "loss_key",
    "n_seeds",
    "test_macro_f1_mean",
    "test_macro_f1_std",
    "test_balanced_accuracy_mean",
    "test_balanced_accuracy_std",
    "test_accuracy_mean",
    "test_accuracy_std",
    "test_ece_uncalibrated_mean",
    "test_ece_uncalibrated_std",
    "test_ece_calibrated_mean",
    "test_ece_calibrated_std",
    "test_nll_uncalibrated_mean",
    "test_nll_uncalibrated_std",
    "test_nll_calibrated_mean",
    "test_nll_calibrated_std",
    "test_brier_uncalibrated_mean",
    "test_brier_uncalibrated_std",
    "test_brier_calibrated_mean",
    "test_brier_calibrated_std",
    "test_mcc_mean",
    "test_mcc_std",
    "test_macro_roc_auc_ovr_mean",
    "test_macro_roc_auc_ovr_std",
    "test_macro_average_precision_ovr_mean",
    "test_macro_average_precision_ovr_std",
    "normal_false_positive_rate_mean",
    "normal_false_positive_rate_std",
    "attack_escape_rate_mean",
    "attack_escape_rate_std",
    "inference_latency_mean_ms_mean",
    "inference_latency_mean_ms_std",
]
comparison_display_columns = [col for col in comparison_display_columns if col in final_model_comparison.columns]

format_dict = {
    column: "{:.4f}"
    for column in comparison_display_columns
    if column.endswith("_mean")
    or column.endswith("_std")
    or column in {"test_macro_f1_mean", "test_macro_f1_std"}
}
if "inference_latency_mean_ms_mean" in format_dict:
    format_dict["inference_latency_mean_ms_mean"] = "{:.3f}"
if "inference_latency_mean_ms_std" in format_dict:
    format_dict["inference_latency_mean_ms_std"] = "{:.3f}"

display(final_model_comparison[comparison_display_columns].style.format(format_dict))

if not aggregated_per_class.empty:
    display(aggregated_per_class)
else:
    print("Aggregated per-class summary is empty.")

if not threshold_summary_df.empty:
    display(threshold_summary_df)
else:
    print("Threshold analysis summary is empty.")

recommendation_display = pd.DataFrame(
    [
        {
            "recommended_model": f"{top_model_key} / {top_loss_key}",
            "primary_metric": primary_metric,
            "top_test_macro_f1_mean": top_score,
            "runner_up_test_macro_f1_mean": runner_up_score,
            "macro_f1_gap": score_gap,
            "results_are_close": results_are_close,
            "tradeoff_note": tradeoff_note,
        }
    ]
)
display(recommendation_display)

print(f"Saved: {EVALUATION_OUTPUT_DIR / 'final_model_comparison.csv'}")
print(f"Saved: {EVALUATION_OUTPUT_DIR / 'aggregated_per_class_summary.csv'}")
print(f"Saved: {EVALUATION_OUTPUT_DIR / 'latency_comparison.csv'}")
print(f"Saved: {EVALUATION_OUTPUT_DIR / 'threshold_analysis.csv'}")
print(f"Saved: {EVALUATION_OUTPUT_DIR / 'model_recommendation.json'}")
print(f"Saved: {EVALUATION_OUTPUT_DIR / 'evaluation_summary.json'}")

if latency_warnings:
    print(f"Latency warnings: {len(latency_warnings)}")
if calibration_warnings:
    print(f"Calibration warnings: {len(calibration_warnings)}")

Seed-count parity check passed: 3 seeds per model


,model_key,loss_key,n_seeds,test_macro_f1_mean,test_macro_f1_std,test_balanced_accuracy_mean,test_balanced_accuracy_std,test_accuracy_mean,test_accuracy_std,test_ece_uncalibrated_mean,test_ece_uncalibrated_std,test_ece_calibrated_mean,test_ece_calibrated_std,test_nll_uncalibrated_mean,test_nll_uncalibrated_std,test_nll_calibrated_mean,test_nll_calibrated_std,test_brier_uncalibrated_mean,test_brier_uncalibrated_std,test_brier_calibrated_mean,test_brier_calibrated_std,test_mcc_mean,test_mcc_std,test_macro_roc_auc_ovr_mean,test_macro_roc_auc_ovr_std,test_macro_average_precision_ovr_mean,test_macro_average_precision_ovr_std,normal_false_positive_rate_mean,normal_false_positive_rate_std,attack_escape_rate_mean,attack_escape_rate_std,inference_latency_mean_ms_mean,inference_latency_mean_ms_std
0,minilm_l6,weighted_ce,3,0.9904,0.0034,0.9947,0.0003,0.9924,0.0008,0.0026,0.0007,0.0042,0.0000,0.0195,0.0013,0.0211,0.0006,0.0108,0.0008,0.0105,0.0005,0.9885,0.0013,0.9999,0.0000,0.9998,0.0000,0.0025,0.0005,0.0027,0.0001,10.772,0.165
1,distilbert,weighted_ce,3,0.9891,0.0003,0.9950,0.0002,0.9928,0.0003,0.0030,0.0003,0.0044,0.0003,0.0176,0.0010,0.0183,0.0009,0.0104,0.0004,0.0098,0.0003,0.9890,0.0004,0.9999,0.0000,0.9999,0.0000,0.0026,0.0010,0.0020,0.0002,10.297,0.364
2,tinybert_bigru_attn,weighted_ce,3,0.9889,0.0004,0.9951,0.0001,0.9928,0.0002,0.0029,0.0001,0.0046,0.0003,0.0180,0.0003,0.0189,0.0001,0.0105,0.0000,0.0100,0.0000,0.9891,0.0003,0.9999,0.0000,0.9998,0.0000,0.0016,0.0005,0.0021,0.0002,16.707,0.104


,model_key,loss_key,label_id,label_name,precision_mean,precision_std,precision_ci95_lower,precision_ci95_upper,recall_mean,recall_std,recall_ci95_lower,recall_ci95_upper,f1_mean,f1_std,f1_ci95_lower,f1_ci95_upper,support_mean,support_std
0,distilbert,weighted_ce,0,Code Injection,0.954028,0.001257,0.952606,0.955450,1.000000,0.000000,1.000000,1.000000,0.976473,0.000658,0.975729,0.977218,837.0,0.0
1,distilbert,weighted_ce,1,Normal,0.991306,0.000934,0.990249,0.992364,0.997449,0.000960,0.996362,0.998535,0.994368,0.000700,0.993576,0.995159,3658.0,0.0
2,distilbert,weighted_ce,2,Other Attacks,0.989260,0.000709,0.988458,0.990062,0.992046,0.000331,0.991671,0.992421,0.990651,0.000214,0.990409,0.990894,6035.0,0.0
3,distilbert,weighted_ce,3,SQL Injection,0.999550,0.000195,0.999330,0.999771,0.990641,0.000446,0.990136,0.991145,0.995076,0.000149,0.994906,0.995245,8975.0,0.0
4,minilm_l6,weighted_ce,0,Code Injection,0.969594,0.026355,0.939771,0.999417,0.999204,0.001380,0.997642,1.000765,0.984043,0.012796,0.969564,0.998523,837.0,0.0
5,minilm_l6,weighted_ce,1,Normal,0.988264,0.000404,0.987808,0.988721,0.997540,0.000473,0.997004,0.998075,0.992880,0.000080,0.992790,0.992970,3658.0,0.0
6,minilm_l6,weighted_ce,2,Other Attacks,0.988342,0.001209,0.986973,0.989710,0.992378,0.002987,0.988997,0.995758,0.990353,0.000880,0.989357,0.991349,6035.0,0.0
7,minilm_l6,weighted_ce,3,SQL Injection,0.999213,0.000195,0.998993,0.999433,0.989749,0.000295,0.989416,0.990083,0.994458,0.000057,0.994394,0.994523,8975.0,0.0
8,tinybert_bigru_attn,weighted_ce,0,Code Injection,0.951844,0.002481,0.949037,0.954651,0.999602,0.000690,0.998821,1.000382,0.975137,0.001177,0.973805,0.976469,837.0,0.0
9,tinybert_bigru_attn,weighted_ce,1,Normal,0.990777,0.000966,0.989683,0.991870,0.998360,0.000473,0.997824,0.998896,0.994553,0.000358,0.994149,0.994958,3658.0,0.0


,model_key,loss_key,threshold,coverage_mean,coverage_std,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,macro_f1_mean,macro_f1_std,normal_false_positive_rate_mean,normal_false_positive_rate_std,attack_escape_rate_mean,attack_escape_rate_std
0,distilbert,weighted_ce,0.5,0.998804,0.000258,0.993293,0.000435,0.995442,0.000358,0.989581,0.000401,0.001825,0.000837,0.001916,0.000255
1,distilbert,weighted_ce,0.7,0.994053,0.000205,0.995547,0.000208,0.996805,0.000203,0.991268,0.000260,0.001191,0.000635,0.001460,0.000318
2,distilbert,weighted_ce,0.8,0.990737,0.000798,0.996619,0.000214,0.997533,0.000224,0.992274,0.000346,0.000925,0.000639,0.000636,0.000127
3,distilbert,weighted_ce,0.9,0.984124,0.001095,0.999496,0.000150,0.999541,0.000176,0.999415,0.000309,0.000742,0.000577,0.000363,0.000098
4,minilm_l6,weighted_ce,0.5,0.999624,0.000129,0.992563,0.000760,0.994826,0.000287,0.990580,0.003323,0.002279,0.000789,0.002736,0.000097
5,minilm_l6,weighted_ce,0.7,0.990737,0.002107,0.996586,0.001298,0.997569,0.000897,0.995424,0.003830,0.001289,0.000425,0.001804,0.000568
6,minilm_l6,weighted_ce,0.8,0.987507,0.000543,0.997993,0.000130,0.998533,0.000066,0.998206,0.000167,0.001298,0.000427,0.001085,0.000221
7,minilm_l6,weighted_ce,0.9,0.982603,0.000334,0.998904,0.000052,0.999112,0.000051,0.998959,0.000163,0.001118,0.000281,0.000620,0.000225
8,tinybert_bigru_attn,weighted_ce,0.5,0.999351,0.000341,0.993177,0.000135,0.995353,0.000144,0.989266,0.000294,0.001368,0.000473,0.001936,0.000132
9,tinybert_bigru_attn,weighted_ce,0.7,0.992669,0.000796,0.996023,0.000255,0.997069,0.000317,0.991452,0.000414,0.000828,0.000276,0.001271,0.000276


,recommended_model,primary_metric,top_test_macro_f1_mean,runner_up_test_macro_f1_mean,macro_f1_gap,results_are_close,tradeoff_note
0,minilm_l6 / weighted_ce,test_macro_f1_mean,0.990434,0.989142,0.001292,True,Top models are close across seeds; prioritize ...


Saved: G:\AI\PDDDD\injection-alert-system\ml_model\notebooks\training done\Final training\results\v3_907k_cleaned_final_confirmatory_weighted_ce_3seed_20260412_035441\evaluation\final_model_comparison.csv
Saved: G:\AI\PDDDD\injection-alert-system\ml_model\notebooks\training done\Final training\results\v3_907k_cleaned_final_confirmatory_weighted_ce_3seed_20260412_035441\evaluation\aggregated_per_class_summary.csv
Saved: G:\AI\PDDDD\injection-alert-system\ml_model\notebooks\training done\Final training\results\v3_907k_cleaned_final_confirmatory_weighted_ce_3seed_20260412_035441\evaluation\latency_comparison.csv
Saved: G:\AI\PDDDD\injection-alert-system\ml_model\notebooks\training done\Final training\results\v3_907k_cleaned_final_confirmatory_weighted_ce_3seed_20260412_035441\evaluation\threshold_analysis.csv
Saved: G:\AI\PDDDD\injection-alert-system\ml_model\notebooks\training done\Final training\results\v3_907k_cleaned_final_confirmatory_weighted_ce_3seed_20260412_035441\evaluation\mode